# MAVRL — memory-augmented, varying-speed obstacle-course flight

Drives the whole pipeline on Modal. Assumes the repo is **already cloned** and this
notebook is running with the repo root as the working directory.

Course: entry window (red = target, blue = decoy) → 0–3 horizontal bars (red = fly
above, blue = fly below) → exit window.

**Cells 4–7 are cheap and each catches a different class of failure — geometry, GL,
visual sanity, noise scale. Run them before anything that costs GPU-hours.**

## 1 · Install

In [ ]:
%uv pip install "mujoco>=3.0" "gymnasium>=0.29" "stable-baselines3>=2.0" \
    torch numpy matplotlib imageio imageio-ffmpeg tensorboard tqdm

## 2 · Runtime

`MUJOCO_GL=egl` **must** be set before mujoco is imported anywhere — that is why it
comes before every other import in the session.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"          # before any mujoco import

import sys, subprocess, pathlib
REPO = pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

import numpy as np, torch, mujoco
print("repo        :", REPO)
print("torch       :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0))
print("mujoco      :", mujoco.__version__)
print("numpy       :", np.__version__)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

## 3 · Volume

Checkpoints and the dataset go on a **mounted volume**, not the container's
ephemeral disk — a restarted container otherwise takes the run with it.

In [ ]:
from pathlib import Path

# Point these at your Modal volume mount if it is not the repo itself.
VOL   = Path(os.environ.get("MAVRL_VOLUME", REPO))
DATA  = VOL / "data"
CKPT  = VOL / "ckpt"
RUNS  = VOL / "runs"
for d in (DATA, CKPT, RUNS):
    d.mkdir(parents=True, exist_ok=True)
    probe = d / ".writable"
    probe.write_text("ok"); probe.unlink()
    print(f"{str(d):50s} writable")

if VOL == REPO:
    print("\nNOTE: writing inside the repo. Bind a Modal volume and set "
          "MAVRL_VOLUME to survive container restarts.")

## 4 · Geometry check (CPU only)

No GL, no GPU. If the course is malformed this fails here in a second rather than
forty minutes into a training run.

In [ ]:
from mavrl.course_world import sample_layout, build_gates, STAGE_STATIONS, ENTRY_Y, STATION_SPACING

rng = np.random.default_rng(0)
for stage, n in enumerate(STAGE_STATIONS):
    lay = sample_layout(rng, n)
    gates = build_gates(lay).gates
    ys = [g.y for g in gates]
    assert ys == sorted(ys) and len(set(ys)) == len(ys), ys
    assert all(g.waypoint()[1] == g.y for g in gates), "waypoint must lie in the gate plane"
    print(f"stage {stage}: {lay.describe():55s} planes={[round(y,2) for y in ys]}")
print("\ngeometry ok")

In [ ]:
# Full unit-test sweep -- same set that runs on a laptop with no simulator.
!python tests/test_mavrl_geometry.py

## 5 · Render smoke test

**The single most valuable cell in this notebook.** If EGL is misconfigured, this
is where you find out.

In [ ]:
import matplotlib.pyplot as plt
from mavrl.course_aviary import CourseAviary
from mavrl.course_world import SemClass, sample_layout
from mavrl import config as C

lay = sample_layout(np.random.default_rng(1), 2)
env = CourseAviary(layout=lay, seed=0, collect_mode=True)
obs, _ = env.reset(seed=0)

print("layout :", lay.describe())
print("image  :", obs["image"].shape, obs["image"].dtype)
print("proprio:", obs["proprio"].shape)
print("render :", C.RENDER_RES, "->", C.IMG_RES)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(obs["image"][..., :3]);                     ax[0].set_title("RGB (noisy input)")
ax[1].imshow(obs["image"][..., 3], cmap="viridis");      ax[1].set_title("depth (uint8)")
ax[2].imshow(obs["depth_m"], cmap="magma");              ax[2].set_title("depth (m, clean)")
ax[3].imshow(obs["seg"], cmap="tab10", vmin=0, vmax=9);  ax[3].set_title("segmentation")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

present = {SemClass(c).name: int((obs["seg"] == c).sum()) for c in np.unique(obs["seg"])}
print("visible classes:", present)

### Policy rate

In [ ]:
t0 = env.step_counter
env.step(np.array([0, 1, 0, 0], dtype=np.float32))
dt = env.step_counter - t0
print(f"sim steps per policy step: {dt} (expected {C.SIM_STEPS_PER_POLICY})")
assert dt == C.SIM_STEPS_PER_POLICY
print(f"policy {C.POLICY_FREQ} Hz | PID {C.CTRL_FREQ} Hz | physics {C.SIM_FREQ} Hz")

## 6 · Layout preview

Eyeball every curriculum stage before committing compute. Red bars are flown
**over**, blue bars **under**.

In [ ]:
import mujoco

def third_person(env, dist=9.0, elev=-12.0, azim=90.0, res=(900, 500)):
    r = mujoco.Renderer(env.model, height=res[1], width=res[0])
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat[:] = [0.0, env.layout.exit_y * 0.5, 1.8]
    cam.distance, cam.elevation, cam.azimuth = dist, elev, azim
    r.update_scene(env.data, camera=cam)
    img = r.render(); r.close()
    return img

rng = np.random.default_rng(7)
fig, axes = plt.subplots(len(STAGE_STATIONS), 1,
                         figsize=(11, 3.0 * len(STAGE_STATIONS)))
for stage, n in enumerate(STAGE_STATIONS):
    lay = sample_layout(rng, n)
    env.set_layout(lay); env.reset()
    axes[stage].imshow(third_person(env))
    axes[stage].set_title(f"stage {stage}: {lay.describe()}", fontsize=10)
    axes[stage].axis("off")
plt.tight_layout(); plt.show()

## 7 · Noise preview

Sensor noise goes on the **encoder input only**; reconstruction targets stay clean
(denoising VAE). Check the magnitude looks like a plausible stereo sensor rather
than static.

In [ ]:
from mavrl.sensor_noise import NoiseConfig, corrupt

env.set_layout(sample_layout(np.random.default_rng(2), 1))
obs, _ = env.reset(seed=3)
clean_d = obs["depth_m"]
noisy_d = corrupt(obs["image"][..., :3], clean_d, NoiseConfig(),
                  np.random.default_rng(0))[1]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(clean_d, cmap="magma"); ax[0].set_title("depth: clean target")
ax[1].imshow(noisy_d, cmap="magma"); ax[1].set_title("depth: noisy encoder input")
im = ax[2].imshow(np.abs(noisy_d - clean_d), cmap="inferno")
ax[2].set_title("|difference|"); plt.colorbar(im, ax=ax[2], fraction=0.046)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

d = np.abs(noisy_d - clean_d)
print(f"median error {np.median(d[d>0])*100:.1f} cm | "
      f"dropped pixels {(noisy_d >= C.DEPTH_MAX).mean():.2%}")

# The proximity weight the SeVAE will apply to reconstruction.
from mavrl.imageproc import proximity_weight
grid = np.linspace(0, 14, 200)
plt.figure(figsize=(6, 3))
plt.plot(grid, proximity_weight(grid)); plt.grid(alpha=.3)
plt.xlabel("distance (m)"); plt.ylabel("reconstruction weight")
plt.title("proximity weight: 1.00 near, 0.31 @6 m, 0.05 floor @12 m")
plt.tight_layout(); plt.show()

env.close()

## 8 · Stage 1 — bootstrap policy

PPO with a **frozen random encoder** on the simplest course. This is the paper's
step 1: it does not need to fly well, only well enough to generate varied data.

Long cells run detached so a dropped connection costs you the log view, not the run.

In [ ]:
import subprocess, signal

def launch(name, args):
    log = RUNS / f"{name}.log"
    cmd = f"nohup python -u {args} > {log} 2>&1 & echo $!"
    pid = subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    print(f"{name}: pid {pid}  ->  {log}")
    print(f"stop with:  !kill {pid}")
    return int(pid), log

def tail(log, n=40):
    print(subprocess.run(["tail", "-n", str(n), str(log)],
                         capture_output=True, text=True).stdout)

def running(pid):
    return subprocess.run(["kill", "-0", str(pid)], capture_output=True).returncode == 0

In [ ]:
pid1, log1 = launch("stage1",
    f"-m mavrl.train_course --stage 0 --frozen-encoder "
    f"--timesteps 1000000 --n-envs 8 --out {RUNS/'stage1'}")

In [ ]:
tail(log1); print("running:", running(pid1))

## 9 · Collect

Two things this does differently from ordinary rollout collection: episodes **do
not end on a missed gate** (the SeVAE needs failure states), and **every episode
gets a fresh layout**, drawn across all stages and all bar heights.

In [ ]:
pid2, log2 = launch("collect",
    f"-m mavrl.collect --episodes 2000 --out {DATA} --split all")

In [ ]:
tail(log2); print("running:", running(pid2))

In [ ]:
from mavrl.dataset import summarize, FrameDataset
from mavrl.course_world import N_SEM_CLASSES

report = summarize(DATA)
print(report)
assert not report["problems"], report["problems"]
assert report["unique_layouts"] > 10, "layouts are not varying across episodes"

counts = FrameDataset(DATA).class_counts(N_SEM_CLASSES)
share = counts / counts.sum()
for i, s in enumerate(share):
    print(f"  {SemClass(i).name:12s} {s:7.4%}")
print("\nBar classes are the rare ones -- that is what the CE class weights are for.")

## 10 · Stage 2 — SeVAE

Proximity-weighted RGB-D reconstruction plus a segmentation head. The seg loss is
deliberately **not** proximity-weighted: that would zero out gradients exactly
where the next bar's colour cue lives.

In [ ]:
!python -u -m mavrl.train_sevae --data {DATA} --out {CKPT}/sevae.pt --epochs 50

In [ ]:
from mavrl.sevae import SeVAE
from mavrl.dataset import FrameDataset

dev = "cuda" if torch.cuda.is_available() else "cpu"
vae = SeVAE().to(dev)
vae.load_state_dict(torch.load(CKPT/"sevae.pt", map_location=dev)["model"]); vae.eval()

batch = next(FrameDataset(DATA).iter_batches(6, np.random.default_rng(0)))
img = torch.as_tensor(batch["image"], device=dev).permute(0, 3, 1, 2)
with torch.no_grad():
    out = vae(img, sample=False)

fig, ax = plt.subplots(4, 6, figsize=(17, 11))
for i in range(6):
    ax[0, i].imshow(batch["image_gt"][i, ..., :3]);                    ax[0, i].set_title("target RGB", fontsize=8)
    ax[1, i].imshow(out.rgb[i].permute(1, 2, 0).cpu().numpy());        ax[1, i].set_title("recon RGB", fontsize=8)
    ax[2, i].imshow(out.depth[i, 0].cpu().numpy(), cmap="viridis");    ax[2, i].set_title("recon depth", fontsize=8)
    ax[3, i].imshow(out.seg_logits[i].argmax(0).cpu().numpy(),
                    cmap="tab10", vmin=0, vmax=9);                     ax[3, i].set_title("pred seg", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()
print("Look for: near geometry sharper than far. That is the proximity weight working.")

## 11 · Stage 3 — memory

`T = 20` is the **aux prediction offset**, not a memory window — an LSTM's hidden
state has no horizon. For attention, `K = 16 < T = 20` on purpose: if the window
still contained step `t-20` the loss would be satisfiable by *copying*, and only
the LSTM would face a real memory task.

The `--mem-tokens 0` run is the control. It should do **worse** on the `past`
term, because a plain 16-step window physically cannot reach `t-20`.

In [ ]:
!python -u -m mavrl.train_memory --data {DATA} --sevae {CKPT}/sevae.pt --memory-type lstm --T 20

In [ ]:
!python -u -m mavrl.train_memory --data {DATA} --sevae {CKPT}/sevae.pt --memory-type attention --mem-tokens 4 --T 20
!python -u -m mavrl.train_memory --data {DATA} --sevae {CKPT}/sevae.pt --memory-type attention --mem-tokens 0 --T 20

### Stage 3b (optional) — the paper's λ sweep

Paper Eq. (2) emits `3 · N_e`, split into past / current / future, with `λ_i ∈ {0,1}` picking which are supervised. The default here is `past,current`; this cell adds `future` back.

`--seq-len` goes up because `future` costs T steps off the **end** of every window as well as the T `past` already costs off the front — at `--seq-len 48` only steps [20, 28) would be supervised. The script prints the range it used.

Expect `future` to sit well above the other two (Fig. 2(b): it is the blurriest of the three), and `past` to be no better for its presence.


In [ ]:
!python -u -m mavrl.train_memory --data {DATA} --sevae {CKPT}/sevae.pt --memory-type lstm --T 20 --aux-segments past,current,future --seq-len 64


In [ ]:
import json
fig, axl = plt.subplots(1, 2, figsize=(13, 4))
for tag in ("lstm", "attention_m4", "attention_m0"):
    f = CKPT / f"memory_{tag}_history.json"
    if not f.exists():
        continue
    h = json.loads(f.read_text())
    axl[0].plot([r["past"] for r in h], label=tag)
    axl[1].plot([r["current"] for r in h], label=tag)
axl[0].set_title("I(t-20) reconstruction  <- the memory test")
axl[1].set_title("I(t) reconstruction  <- should be easy for all")
for a in axl: a.set_xlabel("epoch"); a.set_ylabel("MSE"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print("Expect attention_m0 worst on the left plot. If it is not, the memory "
      "tokens are not doing anything and K<T is not biting.")

### Does the memory actually remember?

The loss curves say which backbone is lower. They do not say whether `Î_{t-20}` is a remembered bar or a smear the mean corridor. Look at the picture.

`past: recon` is built from `z_t` alone — anything recognizable in it is information the recurrent state carried after the frame left the field of view. `current: recon` is nearly free for any backbone and is there as the control.


In [ ]:
from IPython.display import Image, display

for tag in ("lstm", "attention_m4", "attention_m0"):
    f = CKPT / f"memory_{tag}_samples.png"
    if f.exists():
        print(tag); display(Image(str(f)))

# Or regenerate at any time, for any checkpoint:
# !python -m mavrl.visualize --data {DATA} --sevae {CKPT}/sevae.pt --memory {CKPT}/memory_lstm.pt --out samples


## 12 · Behaviour cloning

In [ ]:
!python -u -m mavrl.bc --data {DATA} --sevae {CKPT}/sevae.pt --memory-type lstm --out {CKPT}/bc_lstm.pt --epochs 15

## 13 · Stage 4 — PPO with curriculum

Stages advance on rolling success (≥80 %), and the layout resamples every 10
rollouts within a stage. Every env gets the same layout at the same rollout
boundary.

In [ ]:
pid4, log4 = launch("stage4",
    f"-m mavrl.train_course --curriculum --memory-type lstm "
    f"--init {CKPT}/bc_lstm.pt --timesteps 5000000 --n-envs 8 "
    f"--out {RUNS/'stage4_lstm'}")

In [ ]:
tail(log4, 30); print("running:", running(pid4))

In [ ]:
def plot_run(path, label=None):
    rows = [json.loads(l) for l in (Path(path)/"log.jsonl").read_text().splitlines()]
    ts = [r["timesteps"] for r in rows]
    fig, a = plt.subplots(1, 3, figsize=(16, 3.6))
    a[0].plot(ts, [r["success"] for r in rows]); a[0].set_title("success rate")
    a[1].plot(ts, [r["stage"] for r in rows]);   a[1].set_title("curriculum stage")
    a[2].plot(ts, [r["agv"] for r in rows]);     a[2].set_title("average goal velocity (m/s)")
    for x in a: x.set_xlabel("timesteps"); x.grid(alpha=.3)
    fig.suptitle(label or str(path)); plt.tight_layout(); plt.show()

plot_run(RUNS/"stage4_lstm", "LSTM")

### Memory ablation

Three backbones, ≥3 seeds each. **This is the experiment that decides whether ALD
is worth running** — if attention does not beat LSTM here, there is nothing to
distil.

Honest caveat: the memory demand on this course is gate *counting* (~12 steps),
not the paper's getting-stuck-on-large-obstacles scenario — a thin bar never fills
the field of view. A flat ablation is a plausible real outcome, not necessarily a bug.

In [ ]:
for mem in ("lstm", "attention", "none"):
    for seed in (0, 1, 2):
        launch(f"abl_{mem}_s{seed}",
               f"-m mavrl.train_course --curriculum --memory-type {mem} "
               f"--seed {seed} --timesteps 3000000 --n-envs 8 "
               f"--out {RUNS}/abl_{mem}_s{seed}")

## 14 · ALD

Actor (LSTM, what you would fly) collects; learner (attention) does the RL;
actor is pulled toward it by `KL(π_learner ‖ π_actor)`.

In [ ]:
pid5, log5 = launch("ald",
    f"-m mavrl.ald --actor lstm --learner attention --mem-tokens 4 "
    f"--curriculum --timesteps 3000000 --out {RUNS/'ald'}")

## 15 · Evaluation

Train heights are red {1.20, 1.98} / blue {0.40, 1.20}. The held-out set is red
1.60 / blue 0.80 — never seen in training. A large gap means the heights were
memorized rather than the above/below rule learned.

In [ ]:
!python -u -m mavrl.evaluate --model {RUNS}/stage4_lstm/final.pt --memory-type lstm \
    --episodes 25 --out {RUNS}/eval_lstm.json

In [ ]:
rep = json.loads((RUNS/"eval_lstm.json").read_text())
stages = list(rep["train"])
x = np.arange(len(stages)); w = 0.35
fig, a = plt.subplots(1, 2, figsize=(13, 4))
a[0].bar(x - w/2, [rep["train"][s]["success"] for s in stages], w, label="train heights")
a[0].bar(x + w/2, [rep["eval"][s]["success"] for s in stages], w, label="held-out heights")
a[0].set_ylabel("success rate"); a[0].set_title("generalization")
a[1].bar(x - w/2, [rep["train"][s]["agv"] for s in stages], w, label="train")
a[1].bar(x + w/2, [rep["eval"][s]["agv"] for s in stages], w, label="held-out")
a[1].set_ylabel("AGV (m/s)"); a[1].set_title("average goal velocity")
for ax_ in a:
    ax_.set_xticks(x); ax_.set_xticklabels(stages); ax_.legend(); ax_.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

### Rollout video

In [ ]:
import imageio
from IPython.display import Video
from mavrl.policy import MavrlActorCritic
from mavrl.evaluate import rollout

dev = "cuda" if torch.cuda.is_available() else "cpu"
policy = MavrlActorCritic(memory_type="lstm").to(dev)
policy.load_state_dict(torch.load(RUNS/"stage4_lstm"/"final.pt",
                                  map_location=dev)["policy"], strict=False)
policy.eval()

env = CourseAviary(layout=sample_layout(np.random.default_rng(11), 3), seed=11)
info, frames = rollout(env, policy, dev, record=True)
env.close()
print(info)

out = RUNS / "rollout.mp4"
imageio.mimsave(out, [f.astype(np.uint8) for f in frames], fps=C.POLICY_FREQ)
Video(str(out), embed=True, width=480)

## 16 · Noise ablation

Retrain with sensor noise off. If success then collapses when noise is switched
back on, the policy was leaning on perfect ray-traced depth edges that no real
sensor produces.

In [ ]:
launch("abl_nonoise",
       f"-m mavrl.train_course --curriculum --memory-type lstm --sensor-noise 0 "
       f"--timesteps 3000000 --out {RUNS/'abl_nonoise'}")

In [ ]:
!python -u -m mavrl.evaluate --model {RUNS}/abl_nonoise/final.pt --memory-type lstm \
    --sensor-noise 1.0 --episodes 25 --out {RUNS}/eval_nonoise_withnoise.json

## TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS}